## First method: Inductive Training

1. Import stuff

In [1]:
import os
import json
import numpy as np
import torch
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch_geometric.nn as pyg_nn

from DeepLSD.notebooks.models.lightning.dataset_inductive import GraphDatasetInductive
from models.model_struct_textur_utils import train_inductive, test_inductive, validate_inductive, test_inductive_with_roc, plot_roc_curve

2. import your model (that you defined in the "/models" file)

In [2]:
from models.gat_textural_structural import GAT_TEXTURAL_STRUCTURAL


3. Define your parameters

In [3]:
# Dataset Paramaeters
json_dir          = './json_output/'
roi_output_size = (64,64)
# Model parameters
in_channels_DeepLSD_embedding       = 1280
in_channels                         = 1024
hidden_channels                     = 64
out_channels                        = 64
num_layers                          = 2
dropout                             = 0.1
act                                 = 'relu'
v2                                  = True
jumpingKnowledge_layer              = "lstm"
# Training Prameters
lr                = 1e-2
weight_decay      = 5e-4
batch_size        = 2
epochs            = 10

device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

4. Load data and split it into training/Test dataset

In [ ]:
# Load full dataset
full_dataset = GraphDatasetInductive(json_dir,roi_output_size=roi_output_size)

# Compute split sizes
n_total = len(full_dataset)
n_train = int(0.8 * n_total)
n_val   = int(0.1 * n_total)
n_test  = n_total - n_train - n_val  # ensure it adds up

# Split dataset
train_ds, val_ds, test_ds = random_split(full_dataset, [n_train, n_val, n_test])

# Create data loaders
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)


# test if number of features matches
for i,batch in enumerate(train_loader):
    if i > 1:
        break
    all_roi_features = batch.roi_features
    node_embedings = batch.x
    print(all_roi_features.shape)
    print(node_embedings.shape)



5. Initialize your model

In [5]:
# model, optimizer, loss
model     = GAT_TEXTURAL_STRUCTURAL(in_channels_DeepLSD = in_channels_DeepLSD_embedding,
                                in_channels=in_channels, 
                                hidden_channels=hidden_channels, 
                                out_channels=out_channels,
                                roi_align_embedding_shape = roi_output_size,
                                num_layers=num_layers, 
                                dropout=dropout, 
                                act=act,
                                v2 = True,
                                jk_layer = jumpingKnowledge_layer).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.BCELoss()

6. Train your model

In [ ]:
torch.cuda.empty_cache()
# training loop
for epoch in range(1, epochs+1):

    train_loss = train_inductive(model, train_loader, optimizer, criterion, device=device)

    val_n_acc,  val_e_acc  = validate_inductive(model, criterion, val_loader,  device=device)

    print(f'Epoch {epoch:02d} | '
          f'Loss: {train_loss:.4f} | '
          f'Val Node Loss:  {val_n_acc:.4f} | Val Edge Loss:  {val_e_acc:.4f}')
    
    
torch.save(model.state_dict(), 'model.pth')


7. Test model

In [ ]:
test_n_acc,  test_e_acc, test_n_recall, test_e_recall  = test_inductive(model, test_loader,  device=device)
print(f'Test Node Acc:  {test_n_acc:.4f} | Test Edge Acc:  {test_e_acc:.4f} | Test Node Recall:  {test_n_recall:.4f} | Test Edge Recall:  {test_e_recall:.4f}')

In [ ]:
node_scores, node_labels, edge_scores, edge_labels = test_inductive_with_roc(model, test_loader, device=device)

plot_roc_curve(node_labels, node_scores, title='Node Classification ROC')
plot_roc_curve(edge_labels, edge_scores, title='Edge Classification ROC')
